<a href="https://colab.research.google.com/github/sharyali05/kyrgyz-youtube-visual-audit/blob/main/03_statistical_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Statistical Analysis
### Visual Semiotics and Language Ideologies in Kyrgyz Digital Media

This notebook performs the core statistical comparisons of visual features across language groups.

**Input:** `data/visual_features.csv` (generated by `colab_thumbnail_analysis.ipynb`)

**Output:**
- `results/tables/descriptive_stats.csv`
- `results/tables/kruskal_wallis.csv`
- `results/tables/pairwise_mannwhitney.csv`
- `results/tables/effect_sizes.csv`

---
## Cell 1 — Mount Drive and install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scipy scikit-learn pingouin
print('Done.')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 5.4 MB/s eta 0:00:00
Done.


## Cell 2 — Set paths and load data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
BASE         = Path('/content/drive/MyDrive/kyrgyz-audit')
FEATURES_CSV = BASE / 'data/visual_features.csv'
TABLES_DIR   = BASE / 'results/tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(FEATURES_CSV)
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
df.head()

Loaded 800 rows
Columns: ['video_id', 'url', 'language', 'thumbnail_found', 'mean_saturation', 'mean_luminance', 'saturation_std', 'luminance_std', 'mean_hue', 'red_ratio', 'blue_ratio', 'yellow_ratio', 'text_overlay', 'n_text_chars']


,video_id,url,language,thumbnail_found,mean_saturation,mean_luminance,saturation_std,luminance_std,mean_hue,red_ratio,blue_ratio,yellow_ratio,text_overlay,n_text_chars
0,tZHVXqAkp6E,https://www.youtube.com/watch?v=tZHVXqAkp6E,russian,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HXVORQAiQ5c,https://www.youtube.com/watch?v=HXVORQAiQ5c,russian,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,XHeRLTLGsoA,https://www.youtube.com/watch?v=XHeRLTLGsoA,other,1,91.674450,77.471238,89.650323,66.010091,86.224514,0.338698,0.031632,0.010284,NaN,NaN
3,uXAtWZ7RNAc,https://www.youtube.com/watch?v=uXAtWZ7RNAc,english,1,63.953241,181.672261,93.427056,68.453746,32.431742,0.175827,0.080349,0.027218,NaN,NaN
4,QXglaxD5fag,https://www.youtube.com/watch?v=QXglaxD5fag,russian,1,102.396478,166.251516,103.843390,86.231447,42.340449,0.013877,0.043803,0.116650,NaN,NaN


## Cell 3 — Filter to successful thumbnails only

In [ ]:
# Keep only rows where a thumbnail was successfully fetched
df_valid = df[df['thumbnail_found'] == 1].copy()
print(f'Videos with successful thumbnails: {len(df_valid):,}')
print()
print('Per language group:')
print(df_valid['language'].value_counts().to_string())

Videos with successful thumbnails: 710

Per language group:
language
kyrgyz     182
other      178
russian    177
english    173


## Cell 4 — Define visual features to analyze

In [ ]:
# Features to compare across language groups
FEATURES = {
    'mean_saturation': 'Mean Color Saturation',
    'mean_luminance':  'Mean Luminance (Brightness)',
    'saturation_std':  'Saturation Variability',
    'luminance_std':   'Luminance Variability',
    'red_ratio':       'Red/Orange Pixel Ratio',
    'blue_ratio':      'Blue Pixel Ratio',
    'yellow_ratio':    'Yellow Pixel Ratio',
}

# Core language groups for comparison
CORE_LANGS = ['russian', 'kyrgyz', 'english']
df_core = df_valid[df_valid['language'].isin(CORE_LANGS)].copy()

print(f'Core language groups: {CORE_LANGS}')
print(f'Total videos in core analysis: {len(df_core):,}')

Core language groups: ['russian', 'kyrgyz', 'english']
Total videos in core analysis: 532


---
## Cell 5 — Descriptive statistics
Mean, median, and standard deviation for each feature by language group.

In [ ]:
desc_rows = []

for lang in CORE_LANGS:
    subset = df_core[df_core['language'] == lang]
    for feat, label in FEATURES.items():
        if feat not in subset.columns:
            continue
        vals = subset[feat].dropna()
        desc_rows.append({
            'language':  lang,
            'feature':   feat,
            'label':     label,
            'n':         len(vals),
            'mean':      vals.mean(),
            'median':    vals.median(),
            'std':       vals.std(),
            'min':       vals.min(),
            'max':       vals.max(),
        })

desc_df = pd.DataFrame(desc_rows)
desc_df.to_csv(TABLES_DIR / 'descriptive_stats.csv', index=False)
print('Descriptive stats saved.')
print()

# Print a clean summary table
pivot = desc_df.pivot_table(
    index='feature', columns='language', values='mean'
).round(2)
print('Mean values by language group:')
display(pivot)

Descriptive stats saved.

Mean values by language group:


language,english,kyrgyz,russian
feature,,,
blue_ratio,0.17,0.18,0.14
luminance_std,66.49,70.66,67.24
mean_luminance,158.68,121.57,151.41
mean_saturation,116.24,96.21,109.24
red_ratio,0.25,0.22,0.27
saturation_std,69.72,65.14,67.07
yellow_ratio,0.06,0.05,0.06


---
## Cell 6 — Kruskal-Wallis tests
Non-parametric test of whether distributions differ significantly across all three language groups.
Used instead of ANOVA because visual features are unlikely to be normally distributed.

In [ ]:
from scipy import stats

kw_rows = []

for feat, label in FEATURES.items():
    if feat not in df_core.columns:
        continue

    groups = [
        df_core[df_core['language'] == lang][feat].dropna().values
        for lang in CORE_LANGS
    ]

    # Only run if all groups have data
    if any(len(g) == 0 for g in groups):
        continue

    stat, p = stats.kruskal(*groups)

    kw_rows.append({
        'feature':   feat,
        'label':     label,
        'H_statistic': round(stat, 4),
        'p_value':   round(p, 6),
        'significant': 'Yes' if p < 0.05 else 'No',
    })

kw_df = pd.DataFrame(kw_rows).sort_values('p_value')
kw_df.to_csv(TABLES_DIR / 'kruskal_wallis.csv', index=False)
print('Kruskal-Wallis results saved.')
print()
display(kw_df)

Kruskal-Wallis results saved.



,feature,label,H_statistic,p_value,significant
1,mean_luminance,Mean Luminance (Brightness),60.9456,0.000000,Yes
0,mean_saturation,Mean Color Saturation,20.5705,0.000034,Yes
4,red_ratio,Red/Orange Pixel Ratio,7.2631,0.026475,Yes
5,blue_ratio,Blue Pixel Ratio,7.1160,0.028496,Yes
2,saturation_std,Saturation Variability,6.6078,0.036739,Yes
3,luminance_std,Luminance Variability,5.7118,0.057504,No
6,yellow_ratio,Yellow Pixel Ratio,5.1916,0.074586,No


---
## Cell 7 — Pairwise Mann-Whitney U tests
Post-hoc pairwise comparisons between language groups for each feature.
Bonferroni correction applied to control for multiple comparisons.

In [ ]:
from itertools import combinations

pairs = list(combinations(CORE_LANGS, 2))
n_comparisons = len(pairs) * len(FEATURES)  # Bonferroni denominator
alpha_corrected = 0.05 / n_comparisons

mw_rows = []

for feat, label in FEATURES.items():
    if feat not in df_core.columns:
        continue

    for lang_a, lang_b in pairs:
        a = df_core[df_core['language'] == lang_a][feat].dropna().values
        b = df_core[df_core['language'] == lang_b][feat].dropna().values

        if len(a) == 0 or len(b) == 0:
            continue

        stat, p = stats.mannwhitneyu(a, b, alternative='two-sided')

        # Rank-biserial correlation as effect size
        n1, n2 = len(a), len(b)
        effect_size = 1 - (2 * stat) / (n1 * n2)

        # Direction: which group is higher?
        direction = f'{lang_a} > {lang_b}' if np.median(a) > np.median(b) else f'{lang_b} > {lang_a}'

        mw_rows.append({
            'feature':          feat,
            'label':            label,
            'group_a':          lang_a,
            'group_b':          lang_b,
            'median_a':         round(np.median(a), 3),
            'median_b':         round(np.median(b), 3),
            'direction':        direction,
            'U_statistic':      round(stat, 2),
            'p_value':          round(p, 6),
            'p_bonferroni':     round(p * n_comparisons, 6),
            'effect_size_r':    round(abs(effect_size), 4),
            'significant':      'Yes' if p < alpha_corrected else 'No',
        })

mw_df = pd.DataFrame(mw_rows).sort_values(['feature', 'p_value'])
mw_df.to_csv(TABLES_DIR / 'pairwise_mannwhitney.csv', index=False)
print('Pairwise Mann-Whitney results saved.')
print(f'Bonferroni-corrected alpha: {alpha_corrected:.6f}')
print()

# Show only significant results
sig = mw_df[mw_df['significant'] == 'Yes']
print(f'Significant pairwise comparisons: {len(sig)}')
display(sig[['label', 'group_a', 'group_b', 'median_a', 'median_b', 'direction', 'p_bonferroni', 'effect_size_r']])

Pairwise Mann-Whitney results saved.
Bonferroni-corrected alpha: 0.002381

Significant pairwise comparisons: 3


,label,group_a,group_b,median_a,median_b,direction,p_bonferroni,effect_size_r
3,Mean Luminance (Brightness),russian,kyrgyz,156.683,127.287,russian > kyrgyz,0.000000,0.3606
5,Mean Luminance (Brightness),kyrgyz,english,127.287,166.477,english > kyrgyz,0.000000,0.4508
2,Mean Color Saturation,kyrgyz,english,89.999,114.975,english > kyrgyz,0.000198,0.2720


---
## Cell 8 — Effect size summary
Focus on Russian vs. Kyrgyz comparison specifically, since this is the core research question.

In [ ]:
# Filter to Russian vs Kyrgyz only
rv_k = mw_df[
    ((mw_df['group_a'] == 'russian') & (mw_df['group_b'] == 'kyrgyz')) |
    ((mw_df['group_a'] == 'kyrgyz')  & (mw_df['group_b'] == 'russian'))
].copy()

# Effect size interpretation
def interpret_effect(r):
    if r < 0.1:  return 'negligible'
    elif r < 0.3: return 'small'
    elif r < 0.5: return 'medium'
    else:         return 'large'

rv_k['effect_interpretation'] = rv_k['effect_size_r'].apply(interpret_effect)
rv_k = rv_k.sort_values('effect_size_r', ascending=False)
rv_k.to_csv(TABLES_DIR / 'effect_sizes.csv', index=False)

print('Russian vs. Kyrgyz — Effect Sizes (ranked):')
print()
display(rv_k[['label', 'direction', 'median_a', 'median_b',
              'effect_size_r', 'effect_interpretation', 'significant']])

Russian vs. Kyrgyz — Effect Sizes (ranked):



,label,direction,median_a,median_b,effect_size_r,effect_interpretation,significant
3,Mean Luminance (Brightness),russian > kyrgyz,156.683,127.287,0.3606,medium,Yes
0,Mean Color Saturation,russian > kyrgyz,110.257,89.999,0.1851,small,No
12,Red/Orange Pixel Ratio,russian > kyrgyz,0.242,0.190,0.1652,small,No
15,Blue Pixel Ratio,kyrgyz > russian,0.065,0.132,0.1611,small,No
9,Luminance Variability,kyrgyz > russian,66.882,69.846,0.1089,small,No
6,Saturation Variability,russian > kyrgyz,67.613,64.810,0.0846,negligible,No
18,Yellow Pixel Ratio,russian > kyrgyz,0.027,0.022,0.0750,negligible,No


---
## Cell 9 — Correlation analysis
Are visual features correlated with each other? e.g. do high-saturation thumbnails also tend to be brighter?

In [ ]:
feat_cols = [f for f in FEATURES.keys() if f in df_valid.columns]
corr_matrix = df_valid[feat_cols].corr(method='spearman').round(3)
corr_matrix.to_csv(TABLES_DIR / 'feature_correlations.csv')

print('Spearman correlation matrix:')
display(corr_matrix)

Spearman correlation matrix:


,mean_saturation,mean_luminance,saturation_std,luminance_std,red_ratio,blue_ratio,yellow_ratio
mean_saturation,1.000,0.202,0.400,-0.074,0.228,0.143,0.316
mean_luminance,0.202,1.000,0.133,-0.185,0.034,-0.034,0.231
saturation_std,0.400,0.133,1.000,0.298,0.091,0.109,0.272
luminance_std,-0.074,-0.185,0.298,1.000,0.056,-0.012,0.038
red_ratio,0.228,0.034,0.091,0.056,1.000,-0.328,0.132
blue_ratio,0.143,-0.034,0.109,-0.012,-0.328,1.000,-0.237
yellow_ratio,0.316,0.231,0.272,0.038,0.132,-0.237,1.000


---
## Cell 10 — Plain language summary
Auto-generates a summary of key findings you can paste into your paper draft.

In [ ]:
print('=' * 65)
print('KEY FINDINGS: RUSSIAN vs. KYRGYZ')
print('=' * 65)

for _, row in rv_k.iterrows():
    sig_str = '(significant)' if row['significant'] == 'Yes' else '(not significant)'
    print(f"\n{row['label']}:")
    print(f"  Russian median:  {row['median_a'] if 'russian' in row['group_a'] else row['median_b']:.3f}")
    print(f"  Kyrgyz median:   {row['median_b'] if 'kyrgyz' in row['group_b'] else row['median_a']:.3f}")
    print(f"  Direction:       {row['direction']}")
    print(f"  Effect size (r): {row['effect_size_r']} ({row['effect_interpretation']}) {sig_str}")

print()
print('=' * 65)
print('All tables saved to results/tables/ in Google Drive')
print('=' * 65)

KEY FINDINGS: RUSSIAN vs. KYRGYZ

Mean Luminance (Brightness):
  Russian median:  156.683
  Kyrgyz median:   127.287
  Direction:       russian > kyrgyz
  Effect size (r): 0.3606 (medium) (significant)

Mean Color Saturation:
  Russian median:  110.257
  Kyrgyz median:   89.999
  Direction:       russian > kyrgyz
  Effect size (r): 0.1851 (small) (not significant)

Red/Orange Pixel Ratio:
  Russian median:  0.242
  Kyrgyz median:   0.190
  Direction:       russian > kyrgyz
  Effect size (r): 0.1652 (small) (not significant)

Blue Pixel Ratio:
  Russian median:  0.065
  Kyrgyz median:   0.132
  Direction:       kyrgyz > russian
  Effect size (r): 0.1611 (small) (not significant)

Luminance Variability:
  Russian median:  66.882
  Kyrgyz median:   69.846
  Direction:       kyrgyz > russian
  Effect size (r): 0.1089 (small) (not significant)

Saturation Variability:
  Russian median:  67.613
  Kyrgyz median:   64.810
  Direction:       russian > kyrgyz
  Effect size (r): 0.0846 (negligibl

---
## Interpretation of Findings

### Russian vs. Kyrgyz: Key Takeaways

#### What is statistically significant
Only **luminance (brightness)** reaches statistical significance, with a
medium effect size (r = 0.36, p < 0.05). Russian-language thumbnails are
meaningfully brighter than Kyrgyz ones (median 156.7 vs. 127.3 on a 0–255
scale) — approximately a 23% difference in brightness. This directly
corroborates ethnographic interview data in which participants described
Russian-language content as more "polished," "colorful," and
"professionally produced" compared to Kyrgyz content.

#### What trends in the expected direction but does not reach significance
Color saturation (r = 0.19) and red/orange pixel ratios (r = 0.17) both
trend in the expected direction — Russian > Kyrgyz — but do not reach
statistical significance after Bonferroni correction. Two interpretations
are possible: (1) the effect is real but the current sample size lacks
sufficient power to detect it reliably, or (2) thumbnails are an imperfect
proxy for in-video saturation differences, which may be better captured
through full-frame analysis in future work.

#### The unexpected finding: Kyrgyz thumbnails are more blue
Kyrgyz thumbnails show a substantially higher blue pixel ratio than Russian
ones (median 0.132 vs. 0.065) — a finding that runs counter to the
hypothesis that Kyrgyz content is simply less colorful. Several
interpretations are worth considering:

- **Landscape and sky imagery**: Kyrgyz cultural content frequently
  foregrounds outdoor, rural settings — mountains, sky, steppe — which
  naturally produce higher blue ratios.
- **NGO/institutional aesthetics**: A significant portion of Kyrgyz-language
  children's content is produced by UNICEF and the Aga Khan Foundation
  (e.g., *Keremet Koch*), organizations whose branding and visual design
  conventions favor blue palettes.
- **A distinct rather than deficient aesthetic**: Rather than simply lacking
  the brightness and saturation of Russian content, Kyrgyz thumbnails appear
  to inhabit a different visual register altogether — one associated with
  natural environments and institutional production contexts rather than
  commercial entertainment.

This last point has important theoretical implications. The language ideology
literature cautions against framing heritage language content as merely
inferior to dominant-language content (Woolard, 1998; Irvine & Gal, 2000).
The blue ratio finding suggests that Kyrgyz-language YouTube content is not
simply a lower-quality version of Russian content, but reflects different
production contexts, funding sources, and aesthetic conventions — differences
that community members may nonetheless experience as a quality gap due to the
prestige hierarchies embedded in language ideologies.

#### Limitations to note in the paper
- Analysis is based on thumbnails, which are designed for maximum visual
  appeal and may not represent in-video aesthetics.
- Many videos in the dataset were unavailable at time of thumbnail collection,
  reducing effective sample sizes.
- Bonferroni correction is conservative; some non-significant trends may
  reflect real differences that a larger sample would detect.
- The "other" language group is heterogeneous (Kazakh, unknown, etc.) and
  should be interpreted cautiously in comparative analyses.